In [ ]:
"""
根据hash_html去重后数据，进行html去重脚本

使用方法：
1. 直接调用函数：
   html_dedup(spark, input_hash_base_path, input_html_base_path, output_html_base_path, output_hash_with_domain_base_path, config, index_path)

2. 参数说明：
   - input_hash_base_path: 输入hash_html去重后的数据路径
   - input_html_base_path: 输入原始html数据路径
   - output_html_base_path: 输出html去重后的数据路径
   - output_hash_with_domain_base_path: 输出hash+domain数据路径
   - config: xinghe配置对象
   - index_path: 文件列表路径，用于获取需要处理的文件后缀

3. 处理逻辑：
   - 从index_path读取文件列表，提取文件后缀进行并行处理
   - 对每个文件后缀，分别读取hash去重数据和原始html数据
   - 通过track_id字段进行inner join，保留在hash去重结果中的html数据
   - 从URL提取domain和domain_hash_id字段
   - 生成两个输出：HTML去重数据和Hash+Domain数据

4. 文件路径示例：
   - Hash去重文件: s3://web-parse-hw60p/PJCC-dedup/hash-dedup/v1/20230801/xxx.jsonl.gz
   - 原始HTML文件: s3://cn-common-crawl/jsonl/20230801/xxx.jsonl.gz
   - 输出HTML文件: s3://web-parse-hw60p/PJCC-dedup/html-dedup/v1/20230801/xxx.jsonl.gz
   - 输出Hash+Domain文件: s3://web-parse-hw60p/PJCC-dedup/hash-dedup-with-domain/v1/20230801/xxx.jsonl.gz

5. 数据格式：
   - Hash去重数据: {"sub_path", "hash_html", "track_id", "file_path"}
   - 原始HTML数据: {"sub_path", "html", "track_id", "file_path", "url", ...}
   - 输出HTML数据: 原始HTML数据 + domain相关字段
   - 输出Hash+Domain数据: Hash去重数据 + {"url", "domain", "domain_hash_id"}
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from xinghe.spark import *
from xinghe.s3 import *
import json
import uuid
import traceback
from datetime import datetime
from urllib.parse import urlparse
import xxhash

# 错误日志路径
ERROR_PATH = "s3://qa-huawei/chupei/cc-domain-centric-store/error_logs/"

# 异常日志
def get_s3_doctor(target_theme):
    partition_id = str(uuid.uuid4())
    current_time = datetime.now().strftime("%Y%m%d")
    error_log_path = f"{ERROR_PATH}{target_theme}/{current_time}/{partition_id}.jsonl"
    s3_doc_writer = S3DocWriter(path=error_log_path)
    return s3_doc_writer

# 定义提取domain的UDF
def extract_domain(url):
    if url is None:
        return None
    try:
        hostname = urlparse(url).hostname
        return hostname.lower() if hostname else None
    except Exception as e:
        return None

# 定义计算domain_hash_id的UDF
HASH_COUNT = 10000
def compute_domain_hash(domain):
    if domain is None:
        return None
    return xxhash.xxh64_intdigest(domain) % HASH_COUNT

def process_file_pair(file_suffix, input_hash_base_path, input_html_base_path, output_html_base_path, output_hash_with_domain_base_path):
    """
    处理一对文件的html去重 - 在partition内使用直接文件操作
    """
    
    # 初始化错误日志写入器
    s3_doc_writer = get_s3_doctor("dedup_thr")
    
    try:
        # 构造文件路径
        hash_file_path = f"{input_hash_base_path}/{file_suffix}"
        html_file_path = f"{input_html_base_path}/{file_suffix}"
        output_file_path = f"{output_html_base_path}/{file_suffix}"
        output_hash_with_domain_file_path = f"{output_hash_with_domain_base_path}/{file_suffix}"
        
        print(f"开始处理文件对:")
        print(f"  Hash文件: {hash_file_path}")
        print(f"  HTML文件: {html_file_path}")
        print(f"  输出HTML文件: {output_file_path}")
        print(f"  输出Hash+Domain文件: {output_hash_with_domain_file_path}")
        
        # 读取hash去重后的数据
        hash_data_dict = {}  # track_id -> hash_data
        for zz in read_s3_rows(hash_file_path, use_stream=True):
            try:
                hash_detail = json.loads(zz.value)
                track_id = hash_detail.get("track_id")
                if track_id:
                    hash_data_dict[track_id] = hash_detail
            except Exception as e:
                # 记录解析hash数据的错误
                error_info = {
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                    "traceback": traceback.format_exc(),
                    "input_data": f"file_suffix: {file_suffix}, hash_file: {hash_file_path}, raw_data: {zz.value[:200] if hasattr(zz, 'value') else 'N/A'}",
                    "stage": "parse_hash_data",
                    "timestamp": datetime.now().isoformat()
                }
                s3_doc_writer.write(error_info)
                print(f"解析hash数据失败: {str(e)}")
                continue
        
        print(f"读取hash数据: {len(hash_data_dict)} 条记录")
        
        # 初始化writer - 在循环外创建，避免重复创建
        html_writer = S3DocWriter(output_file_path)
        hash_domain_writer = S3DocWriter(output_hash_with_domain_file_path)
        
        # 读取原始html数据并进行join，同时写入结果
        html_count = 0
        hash_domain_count = 0
        
        for zz in read_s3_rows(html_file_path, use_stream=True):
            try:
                html_detail = json.loads(zz.value)
                track_id = html_detail.get("track_id")
                
                # 通过track_id进行inner join
                if track_id and track_id in hash_data_dict:
                    # 处理URL提取domain
                    url = html_detail.get("url")
                    domain = extract_domain(url)
                    domain_hash_id = compute_domain_hash(domain)
                    
                    # 构造HTML去重结果并立即写入
                    html_result = html_detail.copy()
                    html_result["domain"] = domain
                    html_result["domain_hash_id"] = domain_hash_id
                    html_writer.write(html_result)
                    html_count += 1
                    
                    # 构造Hash+Domain结果并立即写入
                    hash_result = hash_data_dict[track_id].copy()
                    # 去掉sub_path字段
                    if "sub_path" in hash_result:
                        del hash_result["sub_path"]
                    hash_result["url"] = url
                    hash_result["domain"] = domain
                    hash_result["domain_hash_id"] = domain_hash_id
                    hash_domain_writer.write(hash_result)
                    hash_domain_count += 1
                    
            except Exception as e:
                # 记录处理html数据的错误
                error_info = {
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                    "traceback": traceback.format_exc(),
                    "input_data": f"file_suffix: {file_suffix}, html_file: {html_file_path}, raw_data: {zz.value[:200] if hasattr(zz, 'value') else 'N/A'}",
                    "stage": "process_html_data",
                    "timestamp": datetime.now().isoformat()
                }
                s3_doc_writer.write(error_info)
                print(f"处理html数据失败: {str(e)}")
                continue
        
        # 确保所有数据都写入
        html_writer.flush()
        hash_domain_writer.flush()
        
        print(f"Join结果: {html_count} 条记录")
        print(f"成功写入HTML去重文件: {output_file_path}, 记录数: {html_count}")
        print(f"成功写入Hash+Domain文件: {output_hash_with_domain_file_path}, 记录数: {hash_domain_count}")
            
        return html_count
        
    except Exception as e:
        # 记录错误日志
        error_info = {
            "error_type": type(e).__name__,
            "error_message": str(e),
            "traceback": traceback.format_exc(),
            "input_data": f"file_suffix: {file_suffix}",
            "stage": "process_file_pair",
            "timestamp": datetime.now().isoformat()
        }
        s3_doc_writer.write(error_info)
        s3_doc_writer.flush()
        print(f"文件处理失败: {file_suffix}, 错误: {str(e)}")
        return 0

def html_dedup(spark, input_hash_base_path, input_html_base_path, output_html_base_path, output_hash_with_domain_base_path, config, index_path):
    """
    根据hash_html去重后数据，进行html去重
    
    Args:
        spark: SparkSession
        input_hash_base_path: 输入hash_html去重后的数据路径
        input_html_base_path: 输入html数据路径
        output_html_base_path: 输出html去重后的数据路径
        output_hash_with_domain_base_path: 输出hash+domain数据路径
        config: xinghe配置对象
        index_path: 文件列表路径
    """
    print(f"开始处理html去重...")
    print(f"Hash去重数据路径: {input_hash_base_path}")
    print(f"HTML数据路径: {input_html_base_path}")
    print(f"输出HTML路径: {output_html_base_path}")
    print(f"输出Hash+Domain路径: {output_hash_with_domain_base_path}")
    print(f"文件列表路径: {index_path}")
    
    # 读取文件列表
    index_df = read_any_path(spark, index_path, config)
    paths = index_df.rdd.map(lambda row: row.value).collect()
    
    print(f"总文件数: {len(paths)}")
    
    # 并行处理文件
    paths_rdd = spark.sparkContext.parallelize(paths)
    
    # 使用mapPartitions进行批量处理
    def process_partition(file_paths):
        results = []
        for file_path in file_paths:
            # 提取文件后缀
            file_suffix = file_path.split('/')[-2] + '/' + file_path.split('/')[-1]
            result_count = process_file_pair(
                file_suffix, 
                input_hash_base_path, 
                input_html_base_path, 
                output_html_base_path, 
                output_hash_with_domain_base_path
            )
            results.append((file_suffix, result_count))
        return results
    
    # 执行处理
    paths_rdd.mapPartitions(process_partition).count()  # 触发执行
    
    print(f"处理完成!")
    print(f"总文件数: {len(paths)}")
    
    return True


# 使用示例
if __name__ == "__main__":
    
       # 配置
    config = {
        "spark_conf_name": "spark_4",
        "skip_success_check": True,
        "spark.yarn.queue": "pipeline.clean",
        "spark.dynamicAllocation.maxExecutors": 2000, # 控制1万并发
        # "spark.executor.memory": "80g",
        # "spark.executor.memoryOverhead": "40g",  # 增加到40GB
        # "spark.speculation": "true",     # 启用推测执行
        # "maxRecordsPerFile": 200000,      # 增加每文件记录数以减少总文件数
        "output_compression": "gz",
        "skip_output_version": True,
        "skip_output_check": True,
        "spark.sql.shuffle.partitions": "20000",  # 减少分区数
        "spark.default.parallelism": "20000",
        
        # Shuffle 优化配置
        "spark.shuffle.io.maxRetries": "10",  # 增加shuffle重试次数
        "spark.shuffle.io.retryWait": "30s",  # 重试等待时间
        "spark.shuffle.compress": "true",  # 启用shuffle压缩
        "spark.shuffle.spill.compress": "true",  # 启用spill压缩
        
        # 网络和超时配置
        "spark.network.timeout": "3600s",  # 进一步增加网络超时
        "spark.broadcast.timeout": "3600s", 
        "spark.broadcast.compress": "true",
        "spark.rpc.askTimeout": "3600s",   
        "spark.rpc.lookupTimeout": "3600s", 
        "spark.storage.blockManagerSlaveTimeoutMs": "3600000",
        
    }
    spark = new_spark_session("cc_dumps.dedup.thr", config)
    sc = spark.sparkContext
    sc.setLogLevel("ERROR")
    sc
    
    # 示例用法
    input_hash_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash-dedup/v2"
    input_html_base_path = "s3://cn-common-crawl/jsonl"
    output_html_base_path = "s3://web-parse-hw60p/PJCC-dedup/html-dedup/v1"
    output_hash_with_domain_base_path = "s3://web-parse-hw60p/PJCC-dedup/hash-dedup-with-domain/v1"
    index_path = "s3://qa-huawei/chupei/cc-domain-centric-store/PJCC/FILELIST/pjcc-hash-dedup-v2.txt"
    # test config
    # output_html_base_path = "s3://qa-huawei/chupei/cc-domain-centric-store/PJCC-dedup/html-dedup/v1"
    # output_hash_with_domain_base_path = "s3://qa-huawei/chupei/cc-domain-centric-store/PJCC-dedup/hash-dedup-with-domain/v1"
    # index_path = "s3://qa-huawei/chupei/cc-domain-centric-store/PJCC/FILELIST/pjcc-hash-dedup-test100.txt"
    
    
    # 执行去重
    result = html_dedup(spark, input_hash_base_path, input_html_base_path, output_html_base_path, output_hash_with_domain_base_path, config, index_path)
    
    spark.stop() 